# 01. Data Quality Audit & Feature Engineering
**Covers:**
- **Section A:** Data quality checks (duplicate IDs, missing values, impossible timestamps, escalation consistency, spelling variants, partial final week).
- **Section B:** Derived durations, operational flags, calendar features, open ticket age as of 30 Sep, and 14-day repeat-contact detection.
- **Output:** Clean, validated dataset saved to `../data/tickets_clean.csv` for downstream notebooks.

---
## A. Data Quality Checks

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# 1. Resolve project root reliably
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DICT_PATH = RAW_DIR / "rideops_data_dictionary.csv"
DATA_PATH = RAW_DIR / "rideops_support_tickets.csv"

print("=== DATA DICTIONARY ===")
if DICT_PATH.exists():
    display(pd.read_csv(DICT_PATH))

# 2. Load actual support tickets dataset
df = pd.read_csv(DATA_PATH)

print(f"\n=== 0. DATASET OVERVIEW ({DATA_PATH.name}) ===")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
print("Columns:", df.columns.tolist())
display(df.head(3))

# 3. Parse timestamp columns
dt_cols = ["created_at", "first_response_at", "first_closed_at", "final_closed_at"]
for col in dt_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("\n=== A1. DUPLICATE TICKET IDs ===")
if "ticket_id" in df.columns:
    print(f"Duplicate ticket_id count: {df['ticket_id'].duplicated().sum()}")

print("\n=== A2. NULL COUNTS PER COLUMN ===")
null_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2)
})
display(null_summary)

print("\n=== A3. IMPOSSIBLE TIMESTAMPS ===")
if all(c in df.columns for c in dt_cols):
    print("Response before creation:       ", (df["first_response_at"] < df["created_at"]).sum())
    print("First close before creation:    ", (df["first_closed_at"] < df["created_at"]).sum())
    print("Final close before first close: ", (df["final_closed_at"] < df["first_closed_at"]).sum())

print("\n=== A4. ESCALATION CONSISTENCY ===")
if all(c in df.columns for c in ["escalated_to_eng", "eng_handoffs", "eng_outcome"]):
    print("escalated_to_eng breakdown:", df["escalated_to_eng"].value_counts(dropna=False).to_dict())
    esc_yes = df["escalated_to_eng"].astype(str).str.strip().str.lower().isin(["yes", "true", "1"])
    handoff_ge1 = df["eng_handoffs"].fillna(0) >= 1
    outcome_present = df["eng_outcome"].notna()

    inconsistent = df[(esc_yes & (~handoff_ge1 | ~outcome_present)) | (~esc_yes & (handoff_ge1 | outcome_present))]
    print(f"Inconsistent escalation rows: {len(inconsistent)}")
    if len(inconsistent) > 0:
        display(inconsistent[["ticket_id", "escalated_to_eng", "eng_handoffs", "eng_outcome"]].head(10))

print("\n=== A5. SPELLING & CASING VARIANTS ===")
for col in ["category", "agent", "priority", "channel", "client_tier", "eng_outcome"]:
    if col in df.columns:
        vals = sorted(df[col].dropna().unique().tolist())
        print(f"{col} ({len(vals)}): {vals}")

print("\n=== A6. DATE RANGE & PARTIAL LAST WEEK CHECK ===")
if "created_at" in df.columns:
    print(f"Min created_at: {df['created_at'].min()} | Max created_at: {df['created_at'].max()}")
    df["week_start"] = df["created_at"].dt.to_period("W-SUN").apply(lambda r: r.start_time)
    wk_check = df.groupby("week_start").agg(
        tickets=("created_at", "count"),
        first_date=("created_at", lambda s: s.min().strftime("%Y-%m-%d (%a)")),
        last_date=("created_at", lambda s: s.max().strftime("%Y-%m-%d (%a)"))
    )
    display(wk_check)

=== DATA DICTIONARY ===


,column,description
0,ticket_id,Support ticket ID (SUP-xxxxx)
1,created_at,When the client raised the ticket (IST)
2,channel,How the ticket came in
3,client_id / client_name,Fictional corporate client
4,client_tier,Enterprise / Mid-market / SMB
5,category,Issue category tagged by L1 support
6,priority,P1 (critical) to P4 (low)
7,description,One-line summary of the issue
8,assigned_agent,L1 support agent who owns the ticket
9,first_response_at,First reply from L1 support



=== 0. DATASET OVERVIEW (rideops_support_tickets.csv) ===
Rows: 1,444 | Columns: 20
Columns: ['ticket_id', 'created_at', 'channel', 'client_id', 'client_name', 'client_tier', 'category', 'priority', 'description', 'assigned_agent', 'first_response_at', 'escalated_to_eng', 'eng_ticket_id', 'eng_handoffs', 'eng_outcome', 'status', 'first_closed_at', 'final_closed_at', 'reopen_count', 'csat_score']


,ticket_id,created_at,channel,client_id,client_name,client_tier,category,priority,description,assigned_agent,first_response_at,escalated_to_eng,eng_ticket_id,eng_handoffs,eng_outcome,status,first_closed_at,final_closed_at,reopen_count,csat_score
0,SUP-10001,2026-07-02 01:27,Client portal,CL-177,Harbor Health,Mid-market,Live tracking / GPS incorrect,P3,Cab location frozen on map for last 20 mins,Priya D,2026-07-02 02:23,No,NaN,0,NaN,Closed,2026-07-02 07:02,2026-07-02 07:02,0,NaN
1,SUP-10002,2026-07-02 02:53,Email,CL-105,Pinnacle Infotech,Mid-market,Login / access issue,P4,Need access to reports module for new manager,Vikram S,2026-07-02 07:18,No,NaN,0,NaN,Closed,2026-07-02 09:08,2026-07-02 09:08,0,4.0
2,SUP-10003,2026-07-02 04:38,Email,CL-113,Horizon Labs,Enterprise,Live tracking / GPS incorrect,P3,Tracking link not opening for security team,Arjun K,2026-07-02 05:13,No,NaN,0,NaN,Closed,2026-07-02 19:06,2026-07-02 19:06,0,NaN



=== A1. DUPLICATE TICKET IDs ===
Duplicate ticket_id count: 0

=== A2. NULL COUNTS PER COLUMN ===


,dtype,null_count,null_pct
ticket_id,str,0,0.00
created_at,datetime64[us],0,0.00
channel,str,0,0.00
client_id,str,0,0.00
client_name,str,0,0.00
client_tier,str,0,0.00
category,str,0,0.00
priority,str,0,0.00
description,str,0,0.00
assigned_agent,str,0,0.00



=== A3. IMPOSSIBLE TIMESTAMPS ===
Response before creation:        0
First close before creation:     0
Final close before first close:  0

=== A4. ESCALATION CONSISTENCY ===
escalated_to_eng breakdown: {'No': 958, 'Yes': 486}
Inconsistent escalation rows: 0

=== A5. SPELLING & CASING VARIANTS ===
category (10): ['Billing / invoice discrepancy', 'Cab delay / no-show complaint', 'Driver / vehicle compliance docs', 'Employee address / geocode change', 'How-to / feature request', 'Live tracking / GPS incorrect', 'Login / access issue', 'OTP / boarding issue', 'Report export failing', 'Roster / booking not syncing']
priority (4): ['P1', 'P2', 'P3', 'P4']
channel (4): ['Client portal', 'Email', 'Phone', 'WhatsApp']
client_tier (3): ['Enterprise', 'Mid-market', 'SMB']
eng_outcome (6): ['Config issue - returned to L1', 'Duplicate of known issue', 'Fixed', 'Insufficient info - returned to L1', 'Not a bug - how-to', 'Not reproducible']

=== A6. DATE RANGE & PARTIAL LAST WEEK CHECK ===
Min crea

,tickets,first_date,last_date
week_start,,,
2026-06-29,40,2026-07-02 (Thu),2026-07-05 (Sun)
2026-07-06,87,2026-07-06 (Mon),2026-07-12 (Sun)
2026-07-13,87,2026-07-13 (Mon),2026-07-19 (Sun)
2026-07-20,97,2026-07-20 (Mon),2026-07-26 (Sun)
2026-07-27,88,2026-07-27 (Mon),2026-08-02 (Sun)
2026-08-03,107,2026-08-03 (Mon),2026-08-09 (Sun)
2026-08-10,103,2026-08-10 (Mon),2026-08-16 (Sun)
2026-08-17,117,2026-08-17 (Mon),2026-08-23 (Sun)
2026-08-24,114,2026-08-24 (Mon),2026-08-30 (Sun)


---
## B. Derived Columns & Feature Engineering
Using the verified schema from Section A, we compute:
1. **Durations (in hours):** `hrs_to_first_response`, `hrs_to_first_close`, `hrs_to_final_close`, and `resolution_hrs`.
2. **Operational Flags:**
   - `is_escalated`: `escalated_to_eng == 'Yes'`
   - `is_avoidable_esc`: escalated AND `eng_outcome != 'Fixed'`
   - `is_returned_to_l1`: `eng_outcome` in `['Config issue - returned to L1', 'Insufficient info - returned to L1']`
   - `is_reopened`: `reopen_count >= 1`
3. **Calendar & Partial-Week Flags:** `week_start`, `month`, `hour_of_day`, `weekday`, and `is_partial_week` (flagging both `2026-06-29` [4 days] and `2026-09-28` [3 days]).
4. **Open Ticket Age (as of `2026-09-30 23:59:59`):** `open_age_hrs` for tickets where `final_closed_at` is null.
5. **Repeat-Contact Flag:** `is_repeat_contact` = 1 if the same `client_id` raised another ticket in the exact same `category` within 14 days of an earlier ticket.

In [5]:
# Quick check on assigned_agent and status variants before deriving features
for col in ["assigned_agent", "status"]:
    vals = sorted(df[col].dropna().unique().tolist())
    print(f"{col} ({len(vals)}): {vals}")

# Strip any accidental whitespace on string columns
str_cols = ["category", "assigned_agent", "priority", "channel", "client_tier", "eng_outcome", "status"]
for col in str_cols:
    df[col] = df[col].where(df[col].isna(), df[col].astype(str).str.strip())

# 1. Durations in hours
df["hrs_to_first_response"] = (df["first_response_at"] - df["created_at"]).dt.total_seconds() / 3600.0
df["hrs_to_first_close"] = (df["first_closed_at"] - df["created_at"]).dt.total_seconds() / 3600.0
df["hrs_to_final_close"] = (df["final_closed_at"] - df["created_at"]).dt.total_seconds() / 3600.0
df["resolution_hrs"] = df["hrs_to_final_close"]  # Exact time until final resolution (NaN if still open)

# 2. Escalation, Avoidable, Returned-to-L1 & Reopened flags
df["is_escalated"] = (df["escalated_to_eng"] == "Yes").astype(int)

df["is_avoidable_esc"] = (
    (df["is_escalated"] == 1) & (df["eng_outcome"] != "Fixed")
).astype(int)

returned_outcomes = {
    "Config issue - returned to L1",
    "Insufficient info - returned to L1"
}
df["is_returned_to_l1"] = (
    (df["is_escalated"] == 1) & (df["eng_outcome"].isin(returned_outcomes))
).astype(int)

df["is_reopened"] = (df["reopen_count"] >= 1).astype(int)

# 3. Calendar & Partial-Week features
df["week_start"] = df["created_at"].dt.to_period("W-SUN").apply(lambda r: r.start_time)
df["month"] = df["created_at"].dt.to_period("M").astype(str)
df["hour_of_day"] = df["created_at"].dt.hour
df["weekday"] = df["created_at"].dt.day_name()

# Flag partial weeks at start (2026-06-29: 4 days) and end (2026-09-28: 3 days)
partial_weeks = {pd.Timestamp("2026-06-29"), pd.Timestamp("2026-09-28")}
df["is_partial_week"] = df["week_start"].isin(partial_weeks).astype(int)

# 4. Ticket age for open tickets as of 30 Sep 2026
AS_OF_DATE = pd.Timestamp("2026-09-30 23:59:59")
df["is_open"] = df["final_closed_at"].isna().astype(int)
df["open_age_hrs"] = np.where(
    df["is_open"] == 1,
    (AS_OF_DATE - df["created_at"]).dt.total_seconds() / 3600.0,
    np.nan
)

# 5. Repeat-contact flag (same client_id + category within 14 days of an earlier ticket)
df = df.sort_values(["client_id", "category", "created_at", "ticket_id"]).reset_index(drop=True)
df["prev_same_cat_created_at"] = df.groupby(["client_id", "category"])["created_at"].shift(1)
df["days_since_prev_same_cat"] = (df["created_at"] - df["prev_same_cat_created_at"]).dt.total_seconds() / 86400.0
df["is_repeat_contact"] = (
    (df["days_since_prev_same_cat"].notna()) &
    (df["days_since_prev_same_cat"] < 15.0)
).astype(int)

# Restore chronological order by created_at
df = df.sort_values(["created_at", "ticket_id"]).reset_index(drop=True)

# Verification summary of Section B features
print("\n=== SECTION B VERIFICATION SUMMARY ===")
print(f"Total tickets:            {len(df):,}")
print(f"Escalated tickets:        {df['is_escalated'].sum():,} ({df['is_escalated'].mean()*100:.2f}%)")
print(f"Avoidable escalations:    {df['is_avoidable_esc'].sum():,} ({df.loc[df['is_escalated']==1, 'is_avoidable_esc'].mean()*100:.2f}% of escalated)")
print(f"Returned to L1:           {df['is_returned_to_l1'].sum():,} ({df.loc[df['is_escalated']==1, 'is_returned_to_l1'].mean()*100:.2f}% of escalated)")
print(f"Reopened tickets:         {df['is_reopened'].sum():,} ({df['is_reopened'].mean()*100:.2f}%)")
print(f"Open tickets (as of 30 Sep): {df['is_open'].sum():,} (status=='Open': {(df['status']!='Closed').sum()})")
print(f"Repeat contacts (<=14d):  {df['is_repeat_contact'].sum():,} ({df['is_repeat_contact'].mean()*100:.2f}%)")

# Save clean dataset to data/processed/
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = PROCESSED_DIR / "tickets_clean.csv"
df.to_csv(OUT_CSV, index=False)
print(f"\nSaved clean dataset to: {OUT_CSV}")


assigned_agent (8): ['Ananya R', 'Arjun K', 'Karan B', 'Neha P', 'Priya D', 'Rahul T', 'Sneha M', 'Vikram S']
status (3): ['Closed', 'Open - with Engineering', 'Open - with L1']

=== SECTION B VERIFICATION SUMMARY ===
Total tickets:            1,444
Escalated tickets:        486 (33.66%)
Avoidable escalations:    275 (56.58% of escalated)
Returned to L1:           173 (35.60% of escalated)
Reopened tickets:         151 (10.46%)
Open tickets (as of 30 Sep): 48 (status=='Open': 48)
Repeat contacts (<=14d):  288 (19.94%)

Saved clean dataset to: c:\Project\rideops-escalation-gate\data\processed\tickets_clean.csv
